# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ridaeman02/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

I select **Ranking / Scoring** as the ML task type for **Lane 2: Refresh / Content Opportunity Scoring**.

Our goal is to score every page's need for a refresh and then rank them. This allows editors to review the highest opportunity pages first, which makes it a ranking/scoring task type.

In [2]:
import pandas as pd
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
print(f"Total pages across all clients: {len(df)}")
print(f"Average pages per client: {df.groupby('client_id')['content_id'].count().mean():.1f}")

Total pages across all clients: 30000
Average pages per client: 937.5


## 2. Target or proxy

We will predict `is_declining_label`, which is `True` if `trend_direction == 'down'`.

This is an **observed outcome** in the starter dataset because it is calculated from the actual traffic changes (`trend_pct`) over the last 90 days.

In [4]:
df['is_declining_label'] = df['trend_direction'] == 'down'
print("Target distribution (is_declining_label):")
print(df['is_declining_label'].value_counts(normalize=True) * 100)

Target distribution (is_declining_label):
is_declining_label
True     54.206667
False    45.793333
Name: proportion, dtype: float64


## 3. Success metric

The success metric is **Precision@50** (the percentage of true declining pages in the top 50 ranked pages).

This matches the weekly workload capacity of typical content teams. A value of `> 0.70` (getting 35 out of 50 right) is considered good, since the baseline rule only gets `0.24`.

In [6]:
base_rate = (df['trend_direction'] == 'down').mean()
print(f"Base rate of declining pages (random guessing success): {base_rate * 100:.2f}%")

Base rate of declining pages (random guessing success): 54.21%


## 4. The unit of analysis, as a real dataframe

The unit of analysis is **one unique content page for a client**. Each row represents one page, identified by `content_id` and `client_id`.

In [8]:
# Clean and slice the data
slice_df = df[(df['content_age_days'] >= 90) & (df['impressions_90d'] > 0)].copy()
slice_df = slice_df.drop_duplicates(subset=['content_id'])
slice_df['is_declining_label'] = slice_df['trend_direction'] == 'down'

# Select key columns to display the grain
columns_to_show = ['content_id', 'client_id', 'impressions_90d', 'sessions_90d', 'content_age_days', 'is_declining_label']
print(f"DataFrame shape: {slice_df.shape}")
print("\nFirst 3 rows of the unit of analysis:")
print(slice_df[columns_to_show].head(3))

DataFrame shape: (30000, 45)

First 3 rows of the unit of analysis:
             content_id  ... is_declining_label
0  content_304f48230142  ...               True
1  content_a1fb4e703a9e  ...               True
2  content_9aa793d4d895  ...               True

[3 rows x 6 columns]


## 5. Why ML beats a fixed rule here

Fixed rules (like 'refresh if age > 180 days') are too rigid. They do not work well when multiple signals interact.

For example, an old page might still perform well, or a page might have low CTR just because of its average position. ML can automatically weigh and combine signals like clicks, impressions, position, scroll rate, and intent to make better predictions.

In [10]:
# Show correlation of some signals with the target
correlation_cols = ['impressions_90d', 'avg_position', 'ctr', 'content_age_days', 'is_declining_label']
corr = slice_df[correlation_cols].corr()
print("Correlation with is_declining_label:")
print(corr['is_declining_label'].sort_values(ascending=False))

Correlation with is_declining_label:
is_declining_label    1.000000
impressions_90d      -0.018175
avg_position         -0.029035
ctr                  -0.061911
content_age_days     -0.163882
Name: is_declining_label, dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.